In [0]:
import requests
import os

acc_name = "medallionazure"
container = "data"
acc_key = dbutils.secrets.get(scope="azure-storage", key="storage-key")
spark.conf.set(f"fs.azure.account.key.{acc_name}.blob.core.windows.net", acc_key)
bronze_path = f"wasbs://{container}@{acc_name}.blob.core.windows.net/bronze/yellow_taxi"
years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
months = range(1, 13)

for year in years:
    for month in months:
        month_str = str(month).zfill(2)
        file_name = f"yellow_tripdata_{year}-{month_str}.parquet"
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{file_name}"
        
        # Tymczasowa ścieżka na dysku klastra
        local_path = f"/tmp/{file_name}"
        
        try:
            print(f"⬇️ Pobieram z sieci: {file_name}...")
            response = requests.get(url, stream=True)
            
            if response.status_code == 200:
                # Zapisujemy plik lokalnie na klastrze
                with open(local_path, "wb") as f:
                    f.write(response.content)
                
                # Wczytujemy plik lokalny do Sparka i od razu na Azure
                df = spark.read.parquet(f"file:{local_path}")
                
                # Dodajemy timestamp (dobra praktyka!)
                from pyspark.sql.functions import current_timestamp
                df.withColumn("ingestion_timestamp", current_timestamp()) \
                  .write.mode("append").parquet(bronze_path)
                
                # Usuwamy plik tymczasowy, żeby nie zapchać klastra
                os.remove(local_path)
                print(f"✅ Plik {file_name} wylądował w Bronze.")
            else:
                print(f"⚠️ Plik {file_name} nie istnieje na serwerze (Status: {response.status_code})")
                
        except Exception as e:
            print(f"❌ Błąd przy {file_name}: {str(e)}")

print("🚀 KONIEC! Teraz sprawdź rozmiar folderu Bronze.")

In [0]:
# Wczytujemy małą próbkę danych z folderu bronze
df_preview = spark.read.parquet(f"wasbs://data@{acc_name}.blob.core.windows.net/bronze/yellow_taxi")

# Wyświetlamy pierwsze 1000 rekordów
display(df_preview.limit(1000))

In [0]:
# Listujemy zawartość głównego kontenera
display(dbutils.fs.ls(f"wasbs://data@medallionazure.blob.core.windows.net/"))

In [0]:
path = f"wasbs://data@medallionazure.blob.core.windows.net/bronze/yellow_taxi"
size_bytes = sum(f.size for f in dbutils.fs.ls(path) if not f.isDir())
print(f"Rozmiar warstwy Bronze: {size_bytes / (1024**3):.2f} GB")